# Experiment 4: Llama-3.1-70B Generation Quality

**Goal:** Extend model coverage from 3B--8B to 3B--70B with Llama-3.1-70B.

**Setup:** 70B model loaded in NF4 4-bit via bitsandbytes (~40GB VRAM). KV cache is still
extracted in FP16 (the compute dtype), which is methodologically sound---production 70B
models are served quantized.

**Part 1: Generation quality** — 7 compressors x 50 prompts (~4-6 hrs on A100 80GB)

**Part 2: Compression sweep** — 14 compressors x 30 prompts x 4 bandwidths

## Requirements
- **GPU:** A100 80GB High-RAM (70B in NF4 4-bit uses ~40GB + KV cache + activations)
- **HuggingFace token:** Required for gated Llama model access

## Setup
1. Create zip locally (134KB): already at `~/kvshuttle.zip`
2. Set Colab runtime to **A100 High-RAM**
3. Run all cells — upload zip when prompted
4. Paste your HF token for Llama access

In [ ]:
# Cell 1: Check GPU
import torch, gc, os
gc.collect()
torch.cuda.empty_cache()
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name}")
    print(f"Memory: {gpu_mem:.1f} GB")
    print(f"GPU used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
    if gpu_mem < 70:
        print("WARNING: A100 80GB High-RAM required for Llama-3.1-70B in 4-bit.")
        print("         Go to Runtime > Change runtime type > A100 > High-RAM.")
else:
    raise RuntimeError("No GPU detected! Go to Runtime > Change runtime type > GPU")

# Memory fragmentation fix
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# Cell 2: Install dependencies + upload KVShuttle
!pip install -q transformers accelerate bitsandbytes datasets pyyaml tqdm

import os
if not os.path.exists("kvshuttle"):
    from google.colab import files
    print("Upload kvshuttle.zip (~/kvshuttle.zip, ~134KB)")
    uploaded = files.upload()
    !unzip -qo kvshuttle.zip -d KVShuttle
    %cd KVShuttle
    !pip install -q -e .
else:
    print("KVShuttle already installed")

In [ ]:
# Cell 3: HuggingFace login (required for gated Llama-3.1-70B)
from huggingface_hub import login
login()

In [ ]:
# Cell 4: Verify installation
from kvshuttle.models.loader_torch import TORCH_MODEL_REGISTRY
from kvshuttle.compression.registry import list_compressors

assert "llama-3.1-70b" in TORCH_MODEL_REGISTRY, "70B not in registry!"
available = list_compressors()
required = ['identity', 'uniform_int8', 'uniform_int4', 'kivi_2bit',
            'cachegen', 'palu_lr', 'cascade_prune50_int4']
missing = [c for c in required if c not in available]
print(f"Available compressors ({len(available)}): {available}")
if missing:
    print(f"WARNING: Missing compressors: {missing}")
else:
    print("All required compressors available!")
print(f"70B HF ID: {TORCH_MODEL_REGISTRY['llama-3.1-70b']}")

In [ ]:
# Cell 5: Smoke test — load 70B in 4-bit, extract short KV cache
import torch
from kvshuttle.models.loader_torch import load_model_torch
from kvshuttle.models.kv_extractor_torch import extract_kv_cache_torch

model, tokenizer, info = load_model_torch("llama-3.1-70b", load_in_4bit=True)
print(f"Model: {info.num_layers} layers, {info.num_kv_heads} KV heads, head_dim={info.head_dim}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

kv = extract_kv_cache_torch(model, tokenizer, "Hello world, this is a test.")
print(f"KV shape: keys={kv.keys.shape}, values={kv.values.shape}, dtype={kv.keys.dtype}")
assert kv.keys.shape == (80, 8, kv.seq_len, 128), f"Unexpected shape: {kv.keys.shape}"
print(f"GPU after KV extraction: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print("Smoke test PASSED!")

In [ ]:
# Cell 6: Free smoke test model before running experiment subprocess
del model, tokenizer, info, kv
import gc; gc.collect()
torch.cuda.empty_cache()
print(f"GPU free: {(torch.cuda.get_device_properties(0).total_mem - torch.cuda.memory_allocated()) / 1e9:.1f} GB")

## Part 1: Generation Quality (7 compressors x 50 prompts)

Runs as a subprocess. The model is loaded fresh inside the subprocess.
Expected time: ~4-6 hours on A100 80GB.

In [ ]:
# Cell 7: Run generation quality experiment
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m experiments.scripts.run_experiment experiments/configs/generation_quality_torch_70b.yaml

In [ ]:
# Cell 8: Inspect generation quality results
import json
import pandas as pd
from pathlib import Path

results_path = Path("experiments/results/generation_quality_fp16_70b/results.json")

if results_path.exists():
    with open(results_path) as f:
        data = json.load(f)
    print(f"Results: {len(data['results'])} entries")

    df = pd.DataFrame(data['results'])
    cols = ['mean_key_cosine_sim', 'token_agreement', 'perplexity_delta', 'compression_ratio']
    agg_cols = {c: 'mean' for c in cols if c in df.columns}
    if agg_cols:
        summary = df.groupby(['model', 'compressor']).agg(agg_cols).round(4)
        display(summary)
        print("\n=== Key findings (70B) ===")
        for comp in sorted(df['compressor'].unique()):
            if comp == 'identity':
                continue
            cdf = df[df['compressor'] == comp]
            ta = cdf['token_agreement'].mean() if 'token_agreement' in cdf else 'N/A'
            print(f"  70B + {comp}: TA={ta:.3f}" if isinstance(ta, float) else f"  70B + {comp}: TA={ta}")
else:
    print("Results not found. Check experiment output above.")

## Part 2: Compression Sweep (14 compressors x 30 prompts x 4 BW)

Full sweep with all 14 compressors for the model sweep data.
This uses shorter prompts (64-256 tokens) for memory efficiency.

In [ ]:
# Cell 9: Write and run compression sweep config for 70B
import yaml
from pathlib import Path

sweep_config = {
    "experiment": {
        "name": "model_sweep_70b",
        "description": "Compression sweep for Llama-3.1-70B (4-bit loaded)"
    },
    "backend": "torch",
    "load_in_4bit": True,
    "models": ["llama-3.1-70b"],
    "compressors": [
        "identity", "uniform_int8", "uniform_int4", "fp8_e4m3",
        "kivi_2bit", "kvquant_2bit", "cachegen", "palu_lr",
        "topk_prune_50", "cascade_prune50_int4", "palu_int4",
        "mixed_k8v4", "lossless_zstd", "lossless_lz4"
    ],
    "bandwidths_gbps": [1, 10, 50, 100],
    "prompts": {
        "source": "wikitext",
        "count": 30,
        "min_tokens": 64,
        "max_tokens": 256
    },
    "evaluation": {
        "attention_error": True,
        "perplexity": False,
        "token_agreement": False
    },
    "output": {
        "dir": "experiments/results/model_sweep_70b",
        "save_per_layer": False
    }
}

config_path = Path("experiments/configs/model_sweep_70b.yaml")
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, "w") as f:
    yaml.dump(sweep_config, f)

n_results = len(sweep_config['compressors']) * sweep_config['prompts']['count'] * len(sweep_config['bandwidths_gbps'])
print(f"Config: {config_path}")
print(f"Expected results: {n_results}")

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m experiments.scripts.run_experiment {config_path}

In [ ]:
# Cell 10: Download all results
from google.colab import files
from pathlib import Path

for result_dir in ["generation_quality_fp16_70b", "model_sweep_70b"]:
    result_path = Path(f"experiments/results/{result_dir}/results.json")
    if result_path.exists():
        files.download(str(result_path))
        print(f"Downloaded {result_path}")
    else:
        print(f"Not found: {result_path}")

print("\nNext steps:")
print("1. Copy results to experiments/results/generation_quality_fp16_70b/ and model_sweep_70b/")
print("2. Run: python experiments/scripts/merge_fp16_results_v2.py")
print("3. Run: python experiments/scripts/compute_confidence_intervals.py")
print("4. Run: python experiments/scripts/generate_paper_tables.py --full")